# Infer hybrid SHARD — Qwen3.5-2B + adapter phiên 3, khoảng câu [TU, DEN)

Bản chia shard của `infer_hybrid_phien3.ipynb`: 5 acc x 200 câu thay 1 kernel ~9 giờ. Mỗi shard tự
chạy pha 1 (encode dense ~7 phút + rerank 200 câu ~6 phút) cho đúng khoảng của nó rồi sinh. Merge
và dựng submission ở máy dev.

Acc không có quyền model `sunny070306/qwen3-5-2b-base` thì tải `Qwen/Qwen3.5-2B` từ HF, ghim revision.


In [ ]:
# ORCHESTRA-PARAMS
TU, DEN = 0, 200


In [ ]:
import os, sys, shutil, zipfile, subprocess, json, gc
from pathlib import Path

SRC = None
for cand in Path("/kaggle/input").glob("**/*"):
    if (cand.name == "legalqa" and (cand / "bm25.py").exists()
            and "legalqa-v10-src" in str(cand)):
        SRC = cand.parent.parent
        break
assert SRC, "Không thấy codebase — gắn dataset legalqa-v10-src"
REPO = Path("/kaggle/working/repo")
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(SRC, REPO)
os.chdir(REPO)
sys.path.insert(0, "src"); sys.path.insert(0, "scripts")
print("codebase:", (REPO / "COMMIT.txt").read_text().strip())

!pip -q install transformers==5.14.1 peft nltk huggingface_hub
!pip -q uninstall -y torchao

# Adapter phiên 3 được train trên prompt CÓ số hiệu và KHÔNG trần ký tự. Cả hai phải
# khớp, không phải một trong hai: sai bất kỳ cái nào là đưa model vào phân phối nó chưa
# từng thấy, và kết quả tệ đi mà không có dấu vết nào trong log.
os.environ["LEGALQA_SO_HIEU"] = "1"
from legalqa.prompt import CO_SO_HIEU
from build_sft import TOP_N, MAX_CHARS
assert CO_SO_HIEU is True, "prompt đang TẮT số hiệu — sai checkpoint phiên 3"
assert TOP_N == 5 and MAX_CHARS > 10**6, f"codebase CŨ: top_n={TOP_N} max_chars={MAX_CHARS}"
assert (REPO / "scripts/infer.py").exists()
print(f"số hiệu ON | top_n={TOP_N} max_chars={MAX_CHARS:,} (không trần)")

In [ ]:
ch = next((p for p in Path("/kaggle/input").glob("**/chunks.jsonl")), None)
assert ch, "Không thấy chunks.jsonl — gắn dataset legalqa-clean"
CLEAN = Path("/kaggle/working/clean")
CLEAN.mkdir(parents=True, exist_ok=True)
for f in ch.parent.iterdir():
    d = CLEAN / f.name
    if not d.exists():
        d.symlink_to(f)
os.environ["CLEAN_DIR"] = str(CLEAN)
for n in ("chunks.jsonl", "bm25.pkl", "corpus.jsonl"):
    assert (CLEAN / n).exists(), f"thiếu {n} trong legalqa-clean"

# Tập test BTC: dict {id: {question, answer=None}}, KHÔNG phải jsonl của ta.
# `nap_cau_hoi()` nhận cả hai dạng theo đuôi file nên đường dẫn phải giữ đuôi .json.
CAU_HOI = next(Path("/kaggle/input").glob("**/public-official.json"), None)
assert CAU_HOI, "Không thấy public-official.json — gắn dataset official"
_d = json.loads(Path(CAU_HOI).read_text(encoding="utf-8"))
assert len(_d) == 1000, f"tập test có {len(_d)} câu, phải là 1.000"
assert all(v.get("answer") is None for v in _d.values()), "tập test có đáp án?"
print(CAU_HOI, len(_d), "câu")

BASE = next((p.parent for p in Path("/kaggle/input").glob("**/tokenizer.json")
             if "qwen" in str(p).lower() and (p.parent / "config.json").exists()), None)
if BASE is None:
    # sha256 safetensors của model Kaggle sunny070306/qwen3-5-2b-base đo bằng kernel
    # tdwuyn/v10-sha-qwen35 trùng khít HF Qwen/Qwen3.5-2B@15852e8 (aa33250c...). Tải
    # vào /tmp: 4,5 GB không được lọt vào output kernel.
    from huggingface_hub import snapshot_download
    BASE = Path(snapshot_download("Qwen/Qwen3.5-2B", revision="15852e8c16360a2fea060d615a32b45270f8a8fc",
                                  local_dir="/tmp/base"))
    print("model nền tải HF (không gắn Kaggle model)")
assert json.loads((BASE / "config.json").read_text()).get("model_type") == "qwen3_5", "sai model nền"
idx = BASE / "model.safetensors.index.json"
can = sorted(set(json.loads(idx.read_text())["weight_map"].values())) if idx.exists() else []
if [n for n in can if not (BASE / n).exists()]:
    FIX = Path("/kaggle/working/base")
    FIX.mkdir(exist_ok=True)
    for f in BASE.iterdir():
        if not (FIX / f.name).exists():
            (FIX / f.name).symlink_to(f)
    thuc = sorted(BASE.glob("*.safetensors"))
    assert len(thuc) == len(can), f"{len(thuc)} shard thật vs {len(can)} index cần"
    for ten, that in zip(can, thuc):
        if not (FIX / ten).exists():
            (FIX / ten).symlink_to(that)
    BASE = FIX
    print("vá tên shard ->", BASE)

# Adapter phiên 3 (train trên prompt BM25+rerank — lượt 3 nộp 0,5539). Lọc theo tên dataset: gắn nhiều phiên thì glob bắt trúng
# phien2 trước và ta suy luận bằng checkpoint cũ mà không hay.
ADAPTER = next((q.parent for q in Path("/kaggle/input").glob("**/tot_nhat/adapter_config.json")
                if "phien3" in str(q)), None)
assert ADAPTER, "không thấy tot_nhat/ của phiên 3 — gắn dataset legalqa-adapter-phien3"

local = [p.parent for p in Path("/kaggle/input").glob("**/config.json")
         if "XLMRobertaForSequenceClassification" in p.read_text()]
if local:
    RERANK = str(local[0])
else:
    from huggingface_hub import snapshot_download
    RERANK = snapshot_download("AITeamVN/Vietnamese_Reranker")
print("base   ", BASE)
print("adapter", ADAPTER)
print("rerank ", RERANK)

In [ ]:
# Dense: e5-small 118M. §15.8 chốt max_len 256 — `nap_dense()` trong infer.py ép sẵn,
# không truyền từ đây. Tải từ HF vì 118M rẻ hơn một vòng publish dataset.
local = [p.parent for p in Path("/kaggle/input").glob("**/config.json")
         if "e5-small" in str(p).lower() or "multilingual-e5" in str(p).lower()]
if local:
    DENSE = str(local[0])
else:
    from huggingface_hub import snapshot_download
    DENSE = snapshot_download("intfloat/multilingual-e5-small")
print("dense  ", DENSE)

RA = Path("/kaggle/working/out"); RA.mkdir(parents=True, exist_ok=True)
# 200 câu: pha 1 ~15 phút, pha 2 ~200 x 29,3 s = ~1,6 giờ. Trần rộng để hàng đợi GPU/tải HF chậm
# không cắt shard; hết giờ thì du_doan.jsonl vẫn giữ phần đã sinh, không ghi mốc.
subprocess.run([sys.executable, "scripts/infer.py",
    "--cau-hoi", str(CAU_HOI),
    "--model", str(BASE), "--adapter", str(ADAPTER), "--rerank", RERANK,
    "--dense", DENSE,
    "--ra", str(RA), "--tu", str(TU), "--den", str(DEN),
    "--batch-size", "32", "--pool", "50",
    "--max-new", "1024", "--gen-bs", "2",
    "--gio-pha1", "1.0", "--gio-pha2", "9.5",
], check=True)


In [ ]:
ids = [k for k in json.loads(Path(CAU_HOI).read_text(encoding="utf-8"))][TU:DEN]
co = {json.loads(l)["id"] for l in open(RA / "du_doan.jsonl", encoding="utf-8")}
thieu = [k for k in ids if k not in co]
print(f"shard [{TU},{DEN}): {len(ids) - len(thieu)}/{len(ids)} câu")
assert not thieu, f"thiếu {len(thieu)} câu — chạy lại shard, pha 2 nối tiếp"
